# Ollama on Kaggle (T4 x2) for OpenCode / OpenAI-compatible clientsRuns an **Ollama** server on this Kaggle GPU and exposes it publicly through a**Cloudflare quick tunnel** (`...trycloudflare.com`). Copy the printed base URLinto `~/.config/opencode/opencode.jsonc` and use it from the OpenCode CLI onyour machine.## Before running - Kaggle settings1. In the right-side **Settings** panel enable **Internet** (required to   download Ollama and open the tunnel).2. Accelerator: **GPU T4 x2** (Settings > Accelerator).3. Run the cells in order and keep the notebook connected.   The session is capped (~12h) and the tunnel URL changes on every restart.Model: maryasov/qwen2.5-coder-cline:14b-instruct-q8_0 (Q8_0, ~16 GB, fits on 2x T4). If VRAM is tight, switch the pull incell 3 to `maryasov/qwen2.5-coder-cline:14b` (Q4_K_M, ~9 GB).

In [ ]:
# 0/5 Diagnostic start marker (for headless debugging)print("KAGGLE_CELL1_MARKER", flush=True)

In [ ]:
# 1/5 Install Ollama, cloudflared, and GPU deps!apt-get update -qq && apt-get install -y -qq pciutils zstd >/dev/null 2>&1# Check GPU is visible (no driver = CPU-only = too slow for 14B)!nvidia-smi# Install Ollama (no systemd on Kaggle, so we start `serve` manually in cell 2)!curl -fsSL https://ollama.com/install.sh | sh# cloudflared quick tunnel binary!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared!chmod +x /usr/local/bin/cloudflared!echo --- && ollama --version && cloudflared --version

In [ ]:
# 2/5 Start the Ollama server in the backgroundimport os, subprocess, threading, socket, timedef wait_port(port=11434, timeout=120):    deadline = time.time() + timeout    while time.time() < deadline:        try:            with socket.create_connection(("127.0.0.1", port), timeout=2):                return True        except OSError:            time.sleep(1)    return Falsedef pipe_lines(proc, tag):    for line in proc.stdout:        print(f"[{tag}] {line}", end="")env = os.environ.copy()env.update({    "OLLAMA_HOST": "127.0.0.1:11434",    "OLLAMA_ORIGINS": "*",          # CORS for remote callers    "OLLAMA_NUM_CTX": "32768",      # tool-call friendly context    "OLLAMA_KEEP_ALIVE": "-1",      # keep model resident while session lives})proc = subprocess.Popen(    ["/usr/local/bin/ollama", "serve"],    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,)threading.Thread(target=pipe_lines, args=(proc, "OLLAMA"), daemon=True).start()assert wait_port(), "Ollama did not start on 127.0.0.1:11434"print("\nOllama is serving on 127.0.0.1:11434")

In [ ]:
# 3/5 Pull the model (Q8_0 ~16 GB download, takes a few minutes)MODEL = 'maryasov/qwen2.5-coder-cline:14b-instruct-q8_0'!ollama pull {MODEL}!ollama list

In [ ]:
# 4/5 Open a Cloudflare quick tunnel to localhost:11434import subprocess, threading, time, redef wait_port(port=11434, timeout=120):    import socket    deadline = time.time() + timeout    while time.time() < deadline:        try:            with socket.create_connection(("127.0.0.1", port), timeout=2):                return True        except OSError:            time.sleep(1)    return Falseassert wait_port(), "Ollama not listening on 11434"proc = subprocess.Popen(    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:11434", "--no-autoupdate"],    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,)found = {"url": None}def read_output():    for line in proc.stdout:        print(f"[cft] {line}", end="")        m = re.search(r"https://([a-zA-Z0-9-]+\.trycloudflare\.com)", line)        if m and not found["url"]:            found["url"] = "https://" + m.group(1)threading.Thread(target=read_output, daemon=True).start()deadline = time.time() + 90while time.time() < deadline and not found["url"]:    time.sleep(1)url = found["url"]if not url:    raise RuntimeError("Timed out waiting for the tunnel URL. Re-run this cell.")# Persist the URL so a headless `kaggle kernels push` run can pull it backtry:    with open("/kaggle/working/tunnel_url.txt", "w") as f:        f.write(url + "\n")    print("Saved tunnel URL to /kaggle/working/tunnel_url.txt")except OSError:    passprint()print("=" * 70)print("TUNNEL URL       :", url)print("OpenCode baseURL :", f"{url}/v1")print("=" * 70)

In [ ]:
# 5/5 Verify the tunnel and print the exact OpenCode config to useimport json, urllib.request# `url` comes from cell 4 (run cells in order)base = f"{url}/v1"req = urllib.request.Request(base + "/models", method="GET")with urllib.request.urlopen(req, timeout=30) as r:    live = json.loads(r.read())names = [m["id"] for m in live.get("data", [])]print("Live models on the tunnel:", names)assert any("qwen2.5-coder-cline" in n for n in names), "Model not listed yet"config = {    "$schema": "https://opencode.ai/config.json",    "model": f"ollama/{MODEL}",    "provider": {        "ollama": {            "npm": "@ai-sdk/openai-compatible",            "name": "Ollama (Kaggle)",            "options": {"baseURL": base},            "models": {                MODEL: {                    "name": "Qwen2.5-Coder-Cline 14B (Kaggle)",                    "tool_call": True,                    "limit": {"context": 32768, "output": 8192},                }            },        }    },}print()print("=" * 70)print("Paste this into ~/.config/opencode/opencode.jsonc")print("(then restart opencode and re-run it)")print("=" * 70)print(json.dumps(config, indent=2))print("=" * 70)print("NOTE: This run finishes when the cells complete; the cloudflared",      "tunnel stays up until Kaggle reaps the idle session (a while).",      "For a durable session, re-run this notebook and keep the tab open.")